<a href="https://colab.research.google.com/github/Karthik5412/News-Categorizer-Nlp-Model/blob/main/pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv('/content/drive/MyDrive/news model/english_news_dataset.csv')

In [4]:
import re

df['News Categories'] = df['News Categories'].apply(lambda x : re.sub(f'[^a-zAz\s]','',x.lower())).str.split(' ')

In [5]:
needed_cols = ['business', 'world', 'entertainment', 'politics', 'automobile', 'national','science', 'sports','education',
               'startup','healthfitness','technology', 'travel','fasion']

def refeared_final (x):
    if x[0] in needed_cols:
        return x[0]
    else :
        for i in x:
            if i in (needed_cols) :
                return i
        else :
            return ''


df['Final cat'] = df['News Categories'].apply(refeared_final)
df = df[df['Final cat'] != '']
df['Final cat'].value_counts()

,count
Final cat,
business,38412
world,36794
entertainment,30452
politics,28656
national,23350
sports,22125
automobile,21903
science,20875
education,20410


In [6]:
df = df.drop(columns= ['News Categories','Date'])
df

,Headline,Content,Final cat
0,Congress leader Baljinder Singh shot dead at h...,Congress leader Baljinder Singh was shot dead ...,national
1,17-year-old girl preparing for NEET dies by su...,Another NEET aspirant died by suicide in Rajas...,national
2,Hampers to welcome MPs in new Parliament tomor...,In order to mark the first-ever working day of...,national
3,"Only 10% women lawmakers in RS, while only 14%...","Congress President Mallikarjun Kharge, while s...",national
4,"Ganesh temple decorated with notes, coins wort...",The Sri Sathya Ganapathi Temple in Bengaluru a...,national
...,...,...,...
307691,"Tamil Nadu to open 10,000 'CM's pharmacy store...",Tamil Nadu CM MK Stalin has announced that 'Ch...,national
307692,NMC study finds mental health issues prevalent...,One in four MBBS students has a mental disorde...,education
307693,Telangana CM says World Bank will help retire ...,Telangana CM Revanth Reddy said the World Bank...,politics
307694,Dr Gagandeep Kang explores role of AI in vacci...,"Dr Gagandeep Kang, a microbiologist and virolo...",healthfitness


In [7]:
df.to_csv('cleaned_df.csv')

In [12]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
from sklearn.base import BaseEstimator, TransformerMixin

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [13]:
stopwords = set(stopwords.words('english'))
lemetizer = WordNetLemmatizer()

In [20]:
class Preprocessing(BaseEstimator, TransformerMixin) :
    def fit(self, x, y):
        return self

    def healper(self, x):
        data = re.sub(r'[^a-zA-z\s]','',x.lower())

        data = data.split()

        cleaned = [w for w in data if w not in stopwords]

        cleaned = [lemetizer.lemmatize(w) for w in cleaned]

        return cleaned

    def transform(self, x):

        combined = x['Headline'] + ' ' + x['Content']

        return combined.apply(self.healper).tolist()


In [23]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

pipe = Pipeline([
    ('Preprocessing', Preprocessing()),
    ('Tfidf', TfidfVectorizer())
])

In [25]:
pipe.fit_transform(df[['Headline', 'Content']])

TypeError: Preprocessing.fit() missing 1 required positional argument: 'y'